# Day 16 · AM Session — Hypothesis Testing & Confidence Intervals
**PG Diploma · AI-ML & Agentic AI Engineering · IIT Gandhinagar**  
**Artifact 3 — Take-Home Assignment**

---

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import ttest_ind, mannwhitneyu, shapiro, norm, t as t_dist

np.random.seed(42)
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.titlesize': 12})
print('All libraries loaded.')

---
## Part A — Hypothesis Test: Does a New App Onboarding Flow Increase Day-7 Retention? (40%)

### A1. Business Question

A mobile app company changed its onboarding flow for new users.  
**Question:** Does the new onboarding flow lead to higher 7-day retention (percentage of users who come back on Day 7) compared to the old flow?

### A2. Hypotheses

- **H₀ (Null):** The new flow does not improve retention. Mean retention (new) ≤ Mean retention (old).
- **H₁ (Alternative):** The new flow improves retention. Mean retention (new) > Mean retention (old).

**Type:** One-tailed (right-tailed) — we only care if the new flow is *better*, not just different.

**α = 0.05** — Standard for product decisions. A 5% false positive risk is acceptable here because rolling back the UI change is cheap if we're wrong.

### A3. Test Selection

We use an **independent samples t-test** because:
1. The two groups (old flow users vs new flow users) are independent — different users.
2. Sample size is large enough (n=200 per group) for CLT to apply, so we can treat the sample means as approximately Normal.
3. We are comparing means of a continuous metric (retention rate per cohort).

We will verify normality first using the Shapiro-Wilk test.

In [ ]:
# A4 — Simulate data
# Each value = Day-7 retention rate for a cohort of 50 users (so it's a proportion, 0-1)
# Old flow: mean retention ~32%, std ~8%
# New flow: mean retention ~37%, std ~8%
n = 200
old_flow = np.random.normal(loc=0.32, scale=0.08, size=n).clip(0, 1)
new_flow = np.random.normal(loc=0.37, scale=0.08, size=n).clip(0, 1)

print('=== Descriptive Statistics ===')
print(f'Old flow — Mean: {old_flow.mean():.4f},  Std: {old_flow.std():.4f},  n: {len(old_flow)}')
print(f'New flow — Mean: {new_flow.mean():.4f},  Std: {new_flow.std():.4f},  n: {len(new_flow)}')
print(f'Observed difference (new - old): {new_flow.mean() - old_flow.mean():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(old_flow, bins=25, alpha=0.6, color='#3498db', label='Old flow')
axes[0].hist(new_flow, bins=25, alpha=0.6, color='#e74c3c', label='New flow')
axes[0].axvline(old_flow.mean(), color='#2980b9', linestyle='--', linewidth=2, label=f'Old mean={old_flow.mean():.3f}')
axes[0].axvline(new_flow.mean(), color='#c0392b', linestyle='--', linewidth=2, label=f'New mean={new_flow.mean():.3f}')
axes[0].set_title('Day-7 Retention: Old vs New Onboarding')
axes[0].set_xlabel('Retention Rate'); axes[0].set_ylabel('Count'); axes[0].legend(fontsize=9)

axes[1].boxplot([old_flow, new_flow], labels=['Old flow', 'New flow'],
                patch_artist=True,
                boxprops=dict(facecolor='#d6eaf8'),
                medianprops=dict(color='#c0392b', linewidth=2))
axes[1].set_title('Box Plot Comparison'); axes[1].set_ylabel('Retention Rate')

plt.tight_layout()
plt.savefig('A_retention_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# A5 — Normality check + Run the test
print('=== Normality Check (Shapiro-Wilk) ===')
stat_old, p_old = shapiro(old_flow)
stat_new, p_new = shapiro(new_flow)
print(f'Old flow: W={stat_old:.4f}, p={p_old:.4f} -> {"Normal" if p_old > 0.05 else "Not Normal"}')
print(f'New flow: W={stat_new:.4f}, p={p_new:.4f} -> {"Normal" if p_new > 0.05 else "Not Normal"}')

print()
print('=== Independent Samples t-test (one-tailed, right) ===')
# alternative='greater' means H1: new_flow mean > old_flow mean
t_stat, p_two = ttest_ind(new_flow, old_flow, equal_var=False)
p_one = p_two / 2  # one-tailed p-value (since t_stat is positive, this is correct)

print(f'Test statistic (t): {t_stat:.4f}')
print(f'p-value (two-tailed): {p_two:.4f}')
print(f'p-value (one-tailed): {p_one:.4f}')
print()
alpha = 0.05
if p_one < alpha:
    print(f'Decision: REJECT H0 (p={p_one:.4f} < alpha={alpha})')
    print('The new onboarding flow has statistically significantly higher Day-7 retention.')
else:
    print(f'Decision: FAIL TO REJECT H0 (p={p_one:.4f} >= alpha={alpha})')

In [ ]:
# A6 — 95% Confidence Interval for the difference in means
diff = new_flow.mean() - old_flow.mean()
se   = np.sqrt(new_flow.var(ddof=1)/len(new_flow) + old_flow.var(ddof=1)/len(old_flow))
df_ci = len(new_flow) + len(old_flow) - 2
t_crit = t_dist.ppf(0.975, df=df_ci)   # 95% CI uses two-tailed critical value
margin = t_crit * se
ci_lo, ci_hi = diff - margin, diff + margin

print('=== 95% Confidence Interval for (New - Old) ===')
print(f'Point estimate (diff): {diff:.4f}')
print(f'Standard error:        {se:.4f}')
print(f't-critical (df={df_ci}):   {t_crit:.4f}')
print(f'95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]')
print()
print(f'Interpretation: We are 95% confident the true difference in Day-7 retention')
print(f'(new minus old) is between {ci_lo*100:.1f}% and {ci_hi*100:.1f}% percentage points.')
if ci_lo > 0:
    print('Since the entire CI is above zero, the new flow is almost certainly better.')

In [ ]:
# A7 — Effect size (Cohen's d)
pooled_std = np.sqrt((old_flow.var(ddof=1) + new_flow.var(ddof=1)) / 2)
cohens_d   = diff / pooled_std

print("=== Effect Size: Cohen's d ===")
print(f"Cohen's d = {cohens_d:.4f}")
if abs(cohens_d) < 0.2:
    size_label = 'negligible'
elif abs(cohens_d) < 0.5:
    size_label = 'small'
elif abs(cohens_d) < 0.8:
    size_label = 'medium'
else:
    size_label = 'large'
print(f'Effect size magnitude: {size_label}')
print()
pct_lift = (new_flow.mean() - old_flow.mean()) / old_flow.mean() * 100
print(f'Percentage lift: {pct_lift:.1f}%')
print(f'(Retention went from {old_flow.mean()*100:.1f}% to {new_flow.mean()*100:.1f}%)')

### A8 — Stakeholder Interpretation (plain English)

We tested whether our new onboarding screens bring more people back to the app after a week.  
Out of 200 user cohorts on the old flow and 200 on the new one, we saw that the new flow had about **5 percentage points higher** Day-7 retention on average.  
Our statistical test says there is less than a 5% chance this difference happened just by luck — so we can be reasonably confident the new flow is genuinely better.  
The 95% confidence interval tells us the true improvement is likely somewhere between 3 and 7 percentage points, which is meaningful for the business.  
The effect size (Cohen's d ≈ 0.6) is in the "medium" range — not a massive jump, but clearly real and worth shipping.

---
## Part B — Multiple Comparisons & p-hacking (30%)

In [ ]:
# B1 — What is the probability of at least one false positive in 20 tests?
# Under H0, each test has a 5% chance of a false positive
# P(at least one FP) = 1 - P(no FP in any test) = 1 - (1-0.05)^20

alpha   = 0.05
n_tests = 20
prob_at_least_one_fp = 1 - (1 - alpha) ** n_tests
print('=== Multiple Comparison Problem ===')
print(f'Number of simultaneous tests: {n_tests}')
print(f'Alpha per test:               {alpha}')
print(f'P(at least one false positive) = 1 - (1-0.05)^20 = {prob_at_least_one_fp:.4f}  ({prob_at_least_one_fp*100:.1f}%)')
print()
print('This is the FWER (Family-Wise Error Rate).')
print('Running 20 tests at alpha=0.05 gives a ~64% chance of getting at least one')
print('false positive — even if NONE of the tests actually have a real effect!')

In [ ]:
# B2 — Simulation to verify the 64% claim
n_simulations = 10000
false_positive_counts = []

for _ in range(n_simulations):
    # 20 tests where H0 is TRUE (same distribution for control and treatment)
    # So any p < 0.05 is a false positive
    fp_in_this_run = 0
    for _ in range(n_tests):
        ctrl = np.random.normal(0, 1, 100)
        trt  = np.random.normal(0, 1, 100)   # same distribution!
        _, p = ttest_ind(ctrl, trt)
        if p < alpha:
            fp_in_this_run += 1
    false_positive_counts.append(fp_in_this_run)

fp_counts = np.array(false_positive_counts)
simulated_fwer = np.mean(fp_counts >= 1)

print('=== Simulation Results (10,000 simulations, 20 tests each) ===')
print(f'Theoretical FWER:  {prob_at_least_one_fp:.4f}')
print(f'Simulated FWER:    {simulated_fwer:.4f}')
print(f'Match? {"YES" if abs(simulated_fwer - prob_at_least_one_fp) < 0.02 else "NO"}')
print(f'Average false positives per run: {fp_counts.mean():.2f}')

fig, ax = plt.subplots(figsize=(8, 4))
unique, counts = np.unique(fp_counts, return_counts=True)
ax.bar(unique, counts / n_simulations, color='#e74c3c', alpha=0.8, edgecolor='white')
ax.set_title(f'False Positives per Run (20 tests, alpha=0.05, H0 true)\nFWER = {simulated_fwer:.3f} — ~64% of runs have at least 1 FP')
ax.set_xlabel('Number of False Positives in one run of 20 tests')
ax.set_ylabel('Proportion of simulations')
ax.set_xticks(unique)
plt.tight_layout()
plt.savefig('B_multiple_comparisons.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# B3 — Bonferroni Correction
alpha_bonferroni = alpha / n_tests
print('=== Bonferroni Correction ===')
print(f'Original alpha:         {alpha}')
print(f'Number of tests:        {n_tests}')
print(f'Bonferroni alpha:       {alpha_bonferroni}  (= 0.05 / 20)')
print()
print('This means: instead of rejecting H0 when p < 0.05,')
print('we now only reject when p < 0.0025.')
print('This is much stricter — it controls the family-wise error rate at 5%.')
print()

# Verify: FWER after correction
corrected_fwer = 1 - (1 - alpha_bonferroni) ** n_tests
print(f'FWER after Bonferroni correction: {corrected_fwer:.4f}  (safely under {alpha})')

# Show comparison across number of tests
test_counts = np.arange(1, 51)
fwer_uncorrected = 1 - (1 - alpha) ** test_counts
fwer_bonferroni  = 1 - (1 - alpha / test_counts) ** test_counts

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(test_counts, fwer_uncorrected, 'r-',  linewidth=2.5, label='Uncorrected FWER (alpha=0.05 per test)')
ax.plot(test_counts, fwer_bonferroni,  'g--', linewidth=2.5, label='After Bonferroni correction')
ax.axhline(0.05, color='gray', linestyle=':', linewidth=1.5, label='Target FWER = 0.05')
ax.axvline(20, color='#3498db', linestyle='--', linewidth=1.5, label='20 tests (our case)')
ax.set_title('FWER vs Number of Tests — Uncorrected vs Bonferroni')
ax.set_xlabel('Number of simultaneous tests'); ax.set_ylabel('Family-Wise Error Rate')
ax.legend(fontsize=9); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('B_bonferroni.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print(f'Comparison:')
print(f'  Original alpha per test:      0.0500')
print(f'  Bonferroni alpha per test:    {alpha_bonferroni:.4f}')
print(f'  This makes it {1/n_tests}x as easy to reject — much harder to get lucky.')

---
## Part C — Interview Ready (20%)

### C1 — p-value vs Confidence Interval (explained simply)

**p-value:** Answers the yes/no question — *"Could this result have happened by chance?"*  
If p < 0.05, we say "it's unlikely to be just noise, so we reject the null hypothesis."  
It is a single number that gives you a binary decision gate. Useful when you need to decide: ship or don't ship.

**Confidence Interval:** Answers the *"how much?"* question.  
A 95% CI says "the true effect is probably somewhere in this range."  
For example, CI = [2%, 8%] tells you the improvement is somewhere between 2 and 8 percentage points.  
Useful when the business cares about the size of the effect, not just whether it exists.

| Situation | Use | Why |
|---|---|---|
| "Did the change work?" | p-value | Binary decision |
| "By how much did it improve?" | CI | Range of plausible values |
| "Is the effect big enough to matter?" | Both | p can be significant but CI tiny |

> A p-value without a CI is like a smoke alarm without telling you where the fire is.

In [ ]:
# C2 — ab_test() function
from scipy.stats import shapiro, ttest_ind, mannwhitneyu, norm

def ab_test(control, treatment, alpha=0.05):
    """
    Run a two-sample A/B test with automatic test selection.

    Parameters
    ----------
    control   : array-like — metric values for the control group
    treatment : array-like — metric values for the treatment group
    alpha     : float — significance level (default 0.05)

    Returns
    -------
    dict with keys:
        test_used   : str   — which test was selected
        statistic   : float — test statistic
        p_value     : float — p-value
        reject_H0   : bool  — True if p < alpha
        effect_size : float — Cohen's d
        ci_95       : tuple — (lower, upper) 95% CI for the difference in means
    """
    ctrl = np.array(control, dtype=float)
    trt  = np.array(treatment, dtype=float)

    # Step 1: Normality check
    # Shapiro-Wilk works well for n < 5000; use it if sample is small enough
    def is_normal(arr):
        if len(arr) < 3:
            return False   # can't test
        if len(arr) > 5000:
            return True    # CLT applies — assume normal for large samples
        _, p = shapiro(arr)
        return p > 0.05

    ctrl_normal = is_normal(ctrl)
    trt_normal  = is_normal(trt)
    both_normal = ctrl_normal and trt_normal

    print(f'Normality check — Control: {"Normal" if ctrl_normal else "Not Normal"}, '
          f'Treatment: {"Normal" if trt_normal else "Not Normal"}')

    # Step 2: Select test
    if both_normal:
        test_used = 'Welch t-test (independent samples)'
        stat, p_val = ttest_ind(trt, ctrl, equal_var=False)
    else:
        test_used = 'Mann-Whitney U test (non-parametric)'
        stat, p_val = mannwhitneyu(trt, ctrl, alternative='two-sided')

    print(f'Test selected: {test_used}')

    # Step 3: Effect size (Cohen's d)
    pooled_std  = np.sqrt((ctrl.var(ddof=1) + trt.var(ddof=1)) / 2)
    cohens_d    = (trt.mean() - ctrl.mean()) / pooled_std if pooled_std > 0 else 0.0

    # Step 4: 95% CI for difference in means
    diff = trt.mean() - ctrl.mean()
    se   = np.sqrt(trt.var(ddof=1)/len(trt) + ctrl.var(ddof=1)/len(ctrl))
    df   = len(trt) + len(ctrl) - 2
    t_c  = t_dist.ppf(0.975, df=df)
    ci   = (diff - t_c * se, diff + t_c * se)

    return {
        'test_used':   test_used,
        'statistic':   round(stat, 4),
        'p_value':     round(p_val, 4),
        'reject_H0':   bool(p_val < alpha),
        'effect_size': round(cohens_d, 4),
        'ci_95':       (round(ci[0], 4), round(ci[1], 4)),
    }


# ── Test on our retention data ──────────────────────────────────────────
print('=== ab_test() on Retention Data ===')
result = ab_test(old_flow, new_flow, alpha=0.05)
print()
for k, v in result.items():
    print(f'  {k:<15}: {v}')

print()
print('=== ab_test() on small non-normal data (should pick Mann-Whitney) ===')
skewed_ctrl = np.random.exponential(scale=2, size=30)
skewed_trt  = np.random.exponential(scale=2.5, size=30)
result2 = ab_test(skewed_ctrl, skewed_trt, alpha=0.05)
print()
for k, v in result2.items():
    print(f'  {k:<15}: {v}')

### C3 — p=0.04, Effect Size=0.02 — Should We Ship?

Your manager says "p is significant, ship it." Before agreeing, I would ask:

**Q1: What does a 0.02 effect size actually mean for the business?**  
Cohen's d = 0.02 is near-zero. Statistically significant ≠ practically meaningful.  
If the metric is revenue per user, a 0.02% lift on ₹100 average order = ₹0.02 per user.  
With 1 million users that's ₹20,000 — is that worth the engineering, QA, and rollout cost?

**Q2: What is the cost if we're wrong (Type I error)?**  
p=0.04 means there's still a 4% chance this result is a false positive.  
If the new feature causes any harm (slower app, worse experience for some users, higher infra cost), we need to weigh that against an almost-zero benefit.

**Q3: How large was the sample and for how long did we run the test?**  
A very large sample (say, 10 million users) can make a 0.02 effect statistically significant even though it's meaningless in practice. This is the "significant but trivial" trap. Check if the test ran long enough to avoid novelty effects and seasonal bias.

---
## Part D — AI-Augmented Task (10%)

### D1. Prompt Used

> *"Explain Type I and Type II errors in the context of a fraud detection system. What are the real-world consequences of each, and how do you balance them?"*

---

### D2. AI Output (Documented)

**AI Summary:**

The AI explained Type I and Type II errors in the fraud context as follows:

- **Type I Error (False Positive):** The system flags a legitimate transaction as fraud. The customer's card gets blocked even though they did nothing wrong. Consequence: frustrated customer, customer service calls, potential churn. The bank loses a good customer.

- **Type II Error (False Negative):** The system misses actual fraud. A fraudulent transaction goes through. Consequence: financial loss for the bank (or customer), possible regulatory penalty, reputational damage.

- **The tradeoff:** To catch more fraud (reduce Type II errors), you lower your detection threshold — but this also flags more legitimate transactions (increases Type I errors). To stop annoying good customers (reduce Type I), you raise the threshold — but you miss more real fraud (increases Type II). This is exactly the **precision-recall tradeoff**.

- **How to balance:** Use the ROC curve and choose an operating point based on business cost. If fraud losses are catastrophic, optimise for recall (catch every fraudster, accept some false alarms). If customer experience is paramount, optimise for precision.

---

### D3. Evaluation

In [ ]:
# D3 — Simulate the precision-recall tradeoff to verify AI's claim
from sklearn.metrics import precision_score, recall_score
import warnings; warnings.filterwarnings('ignore')

# Simulate fraud detection scores
np.random.seed(42)
n_legit  = 900
n_fraud  = 100
# Fraud scores are generally higher (model gives them higher risk scores)
legit_scores = np.random.beta(2, 8, n_legit)    # low scores, mostly < 0.4
fraud_scores = np.random.beta(6, 3, n_fraud)    # high scores, mostly > 0.5
all_scores   = np.concatenate([legit_scores, fraud_scores])
true_labels  = np.array([0]*n_legit + [1]*n_fraud)

thresholds = np.linspace(0.1, 0.9, 50)
precisions, recalls, type1_rates, type2_rates = [], [], [], []

for thresh in thresholds:
    preds = (all_scores >= thresh).astype(int)
    # Type I = FP rate among negatives (legitimate flagged as fraud)
    fp = ((preds == 1) & (true_labels == 0)).sum()
    tn = ((preds == 0) & (true_labels == 0)).sum()
    # Type II = FN rate among positives (fraud missed)
    fn = ((preds == 0) & (true_labels == 1)).sum()
    tp = ((preds == 1) & (true_labels == 1)).sum()

    type1_rates.append(fp / (fp + tn) if (fp + tn) > 0 else 0)
    type2_rates.append(fn / (fn + tp) if (fn + tp) > 0 else 0)
    p = precision_score(true_labels, preds, zero_division=0)
    r = recall_score(true_labels, preds, zero_division=0)
    precisions.append(p); recalls.append(r)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(thresholds, type1_rates, 'r-',  linewidth=2, label='Type I (FP rate): legit flagged as fraud')
axes[0].plot(thresholds, type2_rates, 'b--', linewidth=2, label='Type II (FN rate): fraud missed')
axes[0].set_title('Type I vs Type II Error Rate as Threshold Changes')
axes[0].set_xlabel('Detection Threshold'); axes[0].set_ylabel('Error Rate')
axes[0].legend(fontsize=9)
axes[0].text(0.5, 0.6, 'Lower threshold\n= more sensitive\n= more Type I errors',
             fontsize=8, color='red', ha='center')
axes[0].text(0.75, 0.3, 'Higher threshold\n= more conservative\n= more Type II errors',
             fontsize=8, color='blue', ha='center')

axes[1].plot(recalls, precisions, 'purple', linewidth=2.5)
axes[1].set_title('Precision-Recall Curve\n(This is the Type I/II tradeoff in disguise)')
axes[1].set_xlabel('Recall (= 1 - Type II rate): How many frauds we catch')
axes[1].set_ylabel('Precision: Of flagged txns, how many are real fraud')
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('D_type_errors_fraud.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== D3 Evaluation of AI Output ===')
print('Did AI correctly explain precision-recall as the Type I/II tradeoff? YES')
print()
print('Verified:')
print('  - As threshold drops, Type I error (FP rate) rises, Type II (FN rate) falls')
print('  - This is identical to the precision-recall tradeoff shown in the curve above')
print('  - Higher recall = catching more fraud = more false alarms = lower precision')
print()
print('Gap in AI output:')
print('  - AI did not mention the AUC-ROC metric as a way to compare systems')
print('  - Did not mention cost-sensitive learning (assigning different costs to FP and FN)')

In [ ]:
# Final summary plot — all key results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Day 16 AM — Hypothesis Testing Summary', fontsize=13, fontweight='bold')

# Plot 1: t-distribution with rejection region
x = np.linspace(-4, 4, 300)
axes[0].plot(x, t_dist.pdf(x, df=398), 'k-', linewidth=2)
t_crit_val = t_dist.ppf(0.95, df=398)
x_reject = np.linspace(t_crit_val, 4, 100)
axes[0].fill_between(x_reject, t_dist.pdf(x_reject, df=398), color='#e74c3c', alpha=0.5, label='Rejection region (5%)')
axes[0].axvline(t_stat, color='#2980b9', linewidth=2, linestyle='--', label=f't-stat={t_stat:.2f}')
axes[0].axvline(t_crit_val, color='#c0392b', linewidth=1.5, linestyle=':', label=f't-crit={t_crit_val:.2f}')
axes[0].set_title('One-tailed t-test\n(Part A)')
axes[0].set_xlabel('t-statistic'); axes[0].legend(fontsize=8)

# Plot 2: CI visualisation
axes[1].errorbar([0], [diff], yerr=[[diff - ci_lo], [ci_hi - diff]],
                 fmt='o', color='#2c3e50', capsize=10, capthick=2, elinewidth=2, markersize=10)
axes[1].axhline(0, color='red', linestyle='--', label='H0: no difference')
axes[1].set_title(f'95% CI for Difference\n[{ci_lo:.3f}, {ci_hi:.3f}]')
axes[1].set_ylabel('Retention difference (new - old)')
axes[1].set_xticks([]); axes[1].legend()

# Plot 3: FWER
tc = np.arange(1, 31)
axes[2].plot(tc, 1-(1-0.05)**tc, 'r-', linewidth=2, label='Uncorrected')
axes[2].plot(tc, 1-(1-0.05/tc)**tc, 'g--', linewidth=2, label='Bonferroni')
axes[2].axhline(0.05, color='gray', linestyle=':', linewidth=1.5)
axes[2].axvline(20, color='#3498db', linestyle='--')
axes[2].set_title('Multiple Comparisons\nFWER vs Number of Tests (Part B)')
axes[2].set_xlabel('Number of tests'); axes[2].set_ylabel('FWER')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('Summary_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('All plots saved.')